# 02 - Extracao dos dados de CNPJ - MG

Neste notebook, executo a extracao dos arquivos publicos de CNPJ da Receita Federal para montar uma base bruta consolidada de empresas ativas em Minas Gerais.

A saida desta etapa sera um CSV em `data/raw/cnpj/cnpj_mg_extracao.csv`, que depois podera ser tratado, segmentado e analisado nas proximas etapas do projeto.

## 1. Definicao dos layouts da Receita Federal

Os arquivos da Receita Federal nao vem com cabecalho. Por isso, defino manualmente os nomes das colunas que serao usados na leitura dos CSVs.

In [3]:
ESTABELECIMENTOS_COLUMNS = [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv",
    "matriz_filial",
    "nome_fantasia",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "nome_cidade_exterior",
    "pais",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio_codigo",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "ddd_fax",
    "fax",
    "email",
    "situacao_especial",
    "data_situacao_especial",
]

EMPRESAS_COLUMNS = [
    "cnpj_basico",
    "nome_empresarial",
    "natureza_juridica",
    "qualificacao_responsavel",
    "capital_social",
    "porte_empresa",
    "ente_federativo_responsavel",
]

MUNICIPIOS_COLUMNS = ["municipio_codigo", "municipio_nome"]
CNAES_COLUMNS = ["cnae_fiscal_principal", "cnae_fiscal_descricao"]

OUTPUT_COLUMNS = [
    "cnpj_basico",
    "cnpj_ordem",
    "cnpj_dv",
    "cnpj",
    "nome_empresarial",
    "nome_fantasia",
    "matriz_filial",
    "situacao_cadastral",
    "data_situacao_cadastral",
    "motivo_situacao_cadastral",
    "data_inicio_atividade",
    "cnae_fiscal_principal",
    "cnae_fiscal_descricao",
    "cnae_fiscal_secundaria",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "cep",
    "uf",
    "municipio_codigo",
    "municipio_nome",
    "ddd_1",
    "telefone_1",
    "ddd_2",
    "telefone_2",
    "email",
    "capital_social",
    "porte_empresa",
    "natureza_juridica",
    "opcao_simples",
    "opcao_mei",
    "fonte_arquivo",
    "data_extracao",
]

len(OUTPUT_COLUMNS)

35

## 2. Funcoes auxiliares

As funcoes abaixo procuram os ZIPs na pasta de downloads e leem os CSVs internos usando o padrao da Receita Federal: separador `;`, codificacao `latin1` e ausencia de cabecalho.

In [4]:
def find_zip_files(keyword: str) -> list[Path]:
    keyword = keyword.lower()
    return sorted(path for path in DOWNLOAD_DIR.glob("*.zip") if keyword in path.name.lower())


def first_file_inside_zip(zip_path: Path) -> str:
    with ZipFile(zip_path) as zip_file:
        file_names = [name for name in zip_file.namelist() if not name.endswith("/")]
    if not file_names:
        raise ValueError(f"ZIP sem arquivo interno: {zip_path}")
    return file_names[0]


def read_zip_csv(zip_path: Path, columns: list[str], chunksize: int | None = None):
    inner_file = first_file_inside_zip(zip_path)
    return pd.read_csv(
        zip_path,
        compression="zip",
        header=None,
        names=columns,
        sep=";",
        dtype=str,
        encoding="latin1",
        chunksize=chunksize,
        low_memory=False,
    )


def read_single_table(keyword: str, columns: list[str]) -> pd.DataFrame:
    files = find_zip_files(keyword)
    if not files:
        raise FileNotFoundError(f"Nenhum ZIP encontrado com '{keyword}' em {DOWNLOAD_DIR}")
    return read_zip_csv(files[0], columns)


def build_cnpj(df: pd.DataFrame) -> pd.Series:
    return df["cnpj_basico"] + df["cnpj_ordem"] + df["cnpj_dv"]

## 3. Conferencia dos arquivos disponiveis

Esta celula lista os arquivos encontrados antes da extracao. Se alguma lista vier vazia, baixe o ZIP correspondente antes de continuar.

In [5]:
available_files = {
    "estabelecimentos": find_zip_files("estabele"),
    "empresas": find_zip_files("empresa"),
    "municipios": find_zip_files("municip"),
    "cnaes": find_zip_files("cnae"),
}

for source_name, files in available_files.items():
    print(f"{source_name}: {len(files)} arquivo(s)")
    for file in files[:5]:
        print(f"  - {file.name}")

estabelecimentos: 10 arquivo(s)
  - Estabelecimentos0.zip
  - Estabelecimentos1.zip
  - Estabelecimentos2.zip
  - Estabelecimentos3.zip
  - Estabelecimentos4.zip
empresas: 10 arquivo(s)
  - Empresas0.zip
  - Empresas1.zip
  - Empresas2.zip
  - Empresas3.zip
  - Empresas4.zip
municipios: 1 arquivo(s)
  - Municipios.zip
cnaes: 1 arquivo(s)
  - Cnaes.zip


## 4. Leitura das tabelas de apoio

Municipios e CNAEs sao tabelas menores. Elas entram como apoio para trocar codigos por nomes e descricoes.

In [6]:
municipios = read_single_table("municip", MUNICIPIOS_COLUMNS)
cnaes = read_single_table("cnae", CNAES_COLUMNS)

municipios.head(), cnaes.head()

(  municipio_codigo           municipio_nome
 0             0001            GUAJARA-MIRIM
 1             0002  ALTO ALEGRE DOS PARECIS
 2             0003              PORTO VELHO
 3             0004                  BURITIS
 4             0005                JI-PARANA,
   cnae_fiscal_principal                              cnae_fiscal_descricao
 0               0111301                                   Cultivo de arroz
 1               0111302                                   Cultivo de milho
 2               0111303                                   Cultivo de trigo
 3               0111399  Cultivo de outros cereais não especificados an...
 4               0112101                        Cultivo de algodão herbáceo)

## 5. Extracao dos estabelecimentos ativos em MG

A tabela de estabelecimentos costuma ser a maior da base. Por isso, a leitura e feita em partes (`chunks`) e ja filtra apenas registros de Minas Gerais com situacao cadastral ativa (`02`).

In [7]:
CHUNKSIZE = 500_000

estabelecimentos_files = find_zip_files("estabele")

filtered_chunks = []
total_rows = 0
total_selected = 0

for zip_path in estabelecimentos_files:
    print(f"Lendo {zip_path.name}...")
    for chunk in read_zip_csv(zip_path, ESTABELECIMENTOS_COLUMNS, chunksize=CHUNKSIZE):
        total_rows += len(chunk)
        selected = chunk[(chunk["uf"] == "MG") & (chunk["situacao_cadastral"] == "02")].copy()
        if selected.empty:
            continue
        selected["fonte_arquivo"] = zip_path.name
        filtered_chunks.append(selected)
        total_selected += len(selected)

estabelecimentos_mg = pd.concat(filtered_chunks, ignore_index=True) if filtered_chunks else pd.DataFrame(columns=ESTABELECIMENTOS_COLUMNS)

print(f"Linhas lidas: {total_rows:,}")
print(f"Estabelecimentos ativos em MG: {total_selected:,}")

estabelecimentos_mg.head()

Lendo Estabelecimentos0.zip...
Lendo Estabelecimentos1.zip...
Lendo Estabelecimentos2.zip...
Lendo Estabelecimentos3.zip...
Lendo Estabelecimentos4.zip...
Lendo Estabelecimentos5.zip...
Lendo Estabelecimentos6.zip...
Lendo Estabelecimentos7.zip...
Lendo Estabelecimentos8.zip...
Lendo Estabelecimentos9.zip...
Linhas lidas: 71,314,053
Estabelecimentos ativos em MG: 3,002,769


,cnpj_basico,cnpj_ordem,cnpj_dv,matriz_filial,nome_fantasia,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,nome_cidade_exterior,pais,...,ddd_1,telefone_1,ddd_2,telefone_2,ddd_fax,fax,email,situacao_especial,data_situacao_especial,fonte_arquivo
0,66617294,0001,03,1,NaN,02,20260120,00,NaN,NaN,...,37,35211013,37,35211680,37,35211013,CGONTIJO@CONTABILIDADEGONTIJO.COM.BR,NaN,NaN,Estabelecimentos0.zip
1,09299839,0005,70,2,NaN,02,20260505,00,NaN,NaN,...,33,33313089,33,33312817,NaN,NaN,FINANCEIRO@SUACASADOPINTOR.COM.BR,NaN,NaN,Estabelecimentos0.zip
2,66617505,0001,08,1,NaN,02,20260505,00,NaN,NaN,...,31,97288011,NaN,NaN,NaN,NaN,VRUBIA659@GMAIL.COM,NaN,NaN,Estabelecimentos0.zip
3,66617569,0001,09,1,NaN,02,20260505,00,NaN,NaN,...,31,99485180,NaN,NaN,NaN,NaN,NDRSER.ADM@GMAIL.COM,NaN,NaN,Estabelecimentos0.zip
4,66617581,0001,13,1,NaN,02,20260505,00,NaN,NaN,...,32,99431919,NaN,NaN,NaN,NaN,E.LUCASRB19@GMAIL.COM,NaN,NaN,Estabelecimentos0.zip


## 6. Leitura das empresas relacionadas

Depois de filtrar os estabelecimentos, leio a tabela de empresas em partes e mantenho apenas os `cnpj_basico` que apareceram na base de MG. Isso reduz o volume antes da juncao.

In [8]:
cnpj_basico_mg = set(estabelecimentos_mg["cnpj_basico"].dropna().unique())
empresas_files = find_zip_files("empresa")

empresa_chunks = []

for zip_path in empresas_files:
    print(f"Lendo {zip_path.name}...")
    for chunk in read_zip_csv(zip_path, EMPRESAS_COLUMNS, chunksize=CHUNKSIZE):
        selected = chunk[chunk["cnpj_basico"].isin(cnpj_basico_mg)].copy()
        if not selected.empty:
            empresa_chunks.append(selected)

empresas_mg = pd.concat(empresa_chunks, ignore_index=True) if empresa_chunks else pd.DataFrame(columns=EMPRESAS_COLUMNS)

print(f"Empresas relacionadas: {len(empresas_mg):,}")
empresas_mg.head()

Lendo Empresas0.zip...
Lendo Empresas1.zip...
Lendo Empresas2.zip...
Lendo Empresas3.zip...
Lendo Empresas4.zip...
Lendo Empresas5.zip...
Lendo Empresas6.zip...
Lendo Empresas7.zip...
Lendo Empresas8.zip...
Lendo Empresas9.zip...
Empresas relacionadas: 2,902,966


,cnpj_basico,nome_empresarial,natureza_juridica,qualificacao_responsavel,capital_social,porte_empresa,ente_federativo_responsavel
0,41273737,VALESKA FERREIRA DE OLIVEIRA SANTOS,2135,50,"1000,00",01,NaN
1,41273800,MARTINS E PALMA LTDA,2062,49,"30000,00",01,NaN
2,41273805,41.273.805 FERNANDO GONCALVES DINIZ,2135,50,"1000,00",01,NaN
3,41273810,RAIMUNDO MOREIRA MONTEIRO 83907580672,2135,50,"1000,00",01,NaN
4,41273840,SIRLENE GOMES DOS SANTOS 05454098612,2135,50,"10000,00",01,NaN


## 7. Consolidacao da base bruta

Nesta etapa, junto estabelecimentos, empresas, municipios e CNAEs. Tambem monto o CNPJ completo e preparo as colunas finais definidas no notebook anterior.

In [9]:
cnpj_mg = (
    estabelecimentos_mg
    .merge(empresas_mg, on="cnpj_basico", how="left")
    .merge(municipios, on="municipio_codigo", how="left")
    .merge(cnaes, on="cnae_fiscal_principal", how="left")
)

cnpj_mg["cnpj"] = build_cnpj(cnpj_mg)
cnpj_mg["opcao_simples"] = pd.NA
cnpj_mg["opcao_mei"] = pd.NA
cnpj_mg["data_extracao"] = pd.Timestamp.today().strftime("%Y-%m-%d")

cnpj_mg = cnpj_mg[OUTPUT_COLUMNS]

cnpj_mg.head()

,cnpj_basico,cnpj_ordem,cnpj_dv,cnpj,nome_empresarial,nome_fantasia,matriz_filial,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,...,ddd_2,telefone_2,email,capital_social,porte_empresa,natureza_juridica,opcao_simples,opcao_mei,fonte_arquivo,data_extracao
0,66617294,0001,03,66617294000103,CONDOMINIO LUCCA RESIDENCE,NaN,1,02,20260120,00,...,37,35211680,CGONTIJO@CONTABILIDADEGONTIJO.COM.BR,"0,00",05,3085,<NA>,<NA>,Estabelecimentos0.zip,2026-05-13
1,09299839,0005,70,09299839000570,L S COMERCIO DE TINTAS LTDA,NaN,2,02,20260505,00,...,33,33312817,FINANCEIRO@SUACASADOPINTOR.COM.BR,"100000,00",05,2062,<NA>,<NA>,Estabelecimentos0.zip,2026-05-13
2,66617505,0001,08,66617505000108,66.617.505 RUBIA VITORIA DE OLIVEIRA,NaN,1,02,20260505,00,...,NaN,NaN,VRUBIA659@GMAIL.COM,"2000,00",01,2135,<NA>,<NA>,Estabelecimentos0.zip,2026-05-13
3,66617569,0001,09,66617569000109,66.617.569 NATALIANA DIAS ROSA,NaN,1,02,20260505,00,...,NaN,NaN,NDRSER.ADM@GMAIL.COM,"1,00",01,2135,<NA>,<NA>,Estabelecimentos0.zip,2026-05-13
4,66617581,0001,13,66617581000113,66.617.581 LUCAS ROSADO BARBOZA,NaN,1,02,20260505,00,...,NaN,NaN,E.LUCASRB19@GMAIL.COM,"3000,00",01,2135,<NA>,<NA>,Estabelecimentos0.zip,2026-05-13


## 8. Validacoes rapidas

Antes de salvar, verifico o volume extraido, a distribuicao por UF e situacao cadastral, alem de uma amostra dos municipios com mais registros.

In [10]:
print(f"Total final de registros: {len(cnpj_mg):,}")
print(f"Total de CNPJs unicos: {cnpj_mg['cnpj'].nunique():,}")

display(cnpj_mg["uf"].value_counts(dropna=False))
display(cnpj_mg["situacao_cadastral"].value_counts(dropna=False))
display(cnpj_mg["municipio_nome"].value_counts(dropna=False).head(10))

Total final de registros: 3,002,769
Total de CNPJs unicos: 3,002,769


uf
MG    3002769
Name: count, dtype: int64

situacao_cadastral
02    3002769
Name: count, dtype: int64

municipio_nome
BELO HORIZONTE          554390
UBERLANDIA              151109
CONTAGEM                112866
JUIZ DE FORA             92618
BETIM                    63449
MONTES CLAROS            60249
UBERABA                  52332
DIVINOPOLIS              47187
GOVERNADOR VALADARES     42843
IPATINGA                 38493
Name: count, dtype: int64

## 9. Salvamento da extracao

A base final e salva no mesmo caminho criado no notebook `01`, substituindo o CSV vazio de cabecalhos pela extracao consolidada.

In [11]:
cnpj_mg.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"Arquivo salvo em: {OUTPUT_CSV}")
print(f"Linhas salvas: {len(cnpj_mg):,}")

Arquivo salvo em: c:\Users\Dell\Desktop\Estudo Dados\b2b-prospect-radar\data\raw\cnpj\cnpj_mg_extracao.csv
Linhas salvas: 3,002,769


## 10. Leitura de conferencia

Por fim, leio uma amostra do CSV salvo para confirmar que o arquivo foi gravado corretamente.

In [12]:
sample = pd.read_csv(OUTPUT_CSV, nrows=5, dtype=str)

print(sample.shape)
sample

(5, 35)


,cnpj_basico,cnpj_ordem,cnpj_dv,cnpj,nome_empresarial,nome_fantasia,matriz_filial,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,...,ddd_2,telefone_2,email,capital_social,porte_empresa,natureza_juridica,opcao_simples,opcao_mei,fonte_arquivo,data_extracao
0,66617294,0001,03,66617294000103,CONDOMINIO LUCCA RESIDENCE,NaN,1,02,20260120,00,...,37,35211680,CGONTIJO@CONTABILIDADEGONTIJO.COM.BR,"0,00",05,3085,NaN,NaN,Estabelecimentos0.zip,2026-05-13
1,09299839,0005,70,09299839000570,L S COMERCIO DE TINTAS LTDA,NaN,2,02,20260505,00,...,33,33312817,FINANCEIRO@SUACASADOPINTOR.COM.BR,"100000,00",05,2062,NaN,NaN,Estabelecimentos0.zip,2026-05-13
2,66617505,0001,08,66617505000108,66.617.505 RUBIA VITORIA DE OLIVEIRA,NaN,1,02,20260505,00,...,NaN,NaN,VRUBIA659@GMAIL.COM,"2000,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13
3,66617569,0001,09,66617569000109,66.617.569 NATALIANA DIAS ROSA,NaN,1,02,20260505,00,...,NaN,NaN,NDRSER.ADM@GMAIL.COM,"1,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13
4,66617581,0001,13,66617581000113,66.617.581 LUCAS ROSADO BARBOZA,NaN,1,02,20260505,00,...,NaN,NaN,E.LUCASRB19@GMAIL.COM,"3000,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13
